# Example-04: Generation of combination (data for twiss from phase)

In [1]:
# Import

import numpy
import pandas
import torch
import yaml

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.statistics import mean, variance
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter
from harmonica.decomposition import Decomposition
from harmonica.model import Model
from harmonica.table import Table

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Range limit can be passed to model on initialization
# This can be used to precompute data used for twiss parameters estimation from phase
# Once computed, model can be saved

In [4]:
# Set model instance

model = Model(path='../config.yaml', limit=8, error=True)
model.save()
del model

# limit -- maximum range limit
# error -- flag to compute model advance errors

# phase advance (and errors) are computed for all pairs combinations

In [5]:
# Load model with precomputed data

model = Model.from_file()
model

Model(path=../config.yaml, model=uncoupled)

In [6]:
# Maximum range limit

model.limit

8

In [7]:
# Range limit endpoints

model.count

# (0, 1)  -- limit=1
# (0, 6)  -- limit=2
# (0, 15) -- limit=3
# (0, 28) -- limit=4
# ...

tensor([  1,   6,  15,  28,  45,  66,  91, 120])

In [8]:
# Combinations

# combo -- index
# index -- index mod total number of locations

probe = 28
combo = model.combo[probe]
index = model.index[probe]

# upto limit=1

print(combo[0:1].cpu().tolist())
print(index[0:1].cpu().tolist())
print()

# upto limit=2

print(combo[0:6].cpu().tolist())
print(index[0:6].cpu().tolist())
print()

# only limit=2

print(combo[1:6].cpu().tolist())
print(index[1:6].cpu().tolist())
print()

[[[28, 27], [28, 29]]]
[[[28, 27], [28, 29]]]

[[[28, 27], [28, 29]], [[28, 26], [28, 27]], [[28, 26], [28, 29]], [[28, 26], [28, 30]], [[28, 29], [28, 30]], [[28, 27], [28, 30]]]
[[[28, 27], [28, 29]], [[28, 26], [28, 27]], [[28, 26], [28, 29]], [[28, 26], [28, 30]], [[28, 29], [28, 30]], [[28, 27], [28, 30]]]

[[[28, 26], [28, 27]], [[28, 26], [28, 29]], [[28, 26], [28, 30]], [[28, 29], [28, 30]], [[28, 27], [28, 30]]]
[[[28, 26], [28, 27]], [[28, 26], [28, 29]], [[28, 26], [28, 30]], [[28, 29], [28, 30]], [[28, 27], [28, 30]]]



In [9]:
# Phase advance and phase advance error (value for each combination for all locations)
# [[i, j], [i, k]]

print(model.fx_ij.shape)
print(model.fx_ik.shape)
print(model.sigma_fx_ij.shape)
print(model.sigma_fx_ik.shape)

torch.Size([120, 59])
torch.Size([120, 59])
torch.Size([120, 59])
torch.Size([120, 59])


In [10]:
# Compare with direct computation

probe = 28
value = 0
combo = model.combo[probe, value]
index = model.index[probe, value]

print(combo.cpu().tolist())
print(index.cpu().tolist())

phase, _ = Decomposition.phase_advance(*combo.T, model.nux, model.fx)
print(phase.cpu().tolist())

print([model.fx_ij[value, probe].cpu().item(), model.fx_ik[value, probe].cpu().item()])

[[28, 27], [28, 29]]
[[28, 27], [28, 29]]
[-1.3339903615135995, 1.33399036151393]
[-1.3339903615135995, 1.33399036151393]


In [11]:
# Given a range limit, other monitor locations within this limit are stored in table
# Print other locations for location zero
# Note, virtual locations are not included as other

model.table[0]

[-10, -9, -8, -7, -6, -4, -3, -2, 1, 3, 4, 5, 6, 7, 8, 9]